In [ ]:
%%bash
if [ -d "./cifar10_data/cifar-10-batches-py" ] && [ -f "./cifar10_data/cifar-10-batches-py/batches.meta" ]; then
    echo "Dataset already exists, skipping download."
else
    echo "Dataset not found, downloading..."
    !kaggle datasets download -d pankrzysiu/cifar10-python --force
    !unzip -q cifar10-python.zip -d ./cifar10_data
fi

In [ ]:
import pickle
import numpy as np
import os


def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict


data_dir = './cifar10_data/cifar-10-batches-py/'

x_train_list = []
y_train_list = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    batch = unpickle(batch_path)
    x_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])

x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

test_batch = unpickle(os.path.join(data_dir, 'test_batch'))
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

meta_batch = unpickle(os.path.join(data_dir, 'batches.meta'))
classes = [b.decode('utf-8') for b in meta_batch[b'label_names']]

print(f"Training features shape: {x_train.shape}")
print(f"Training labels shape:   {y_train.shape}")
print(f"Testing features shape:  {x_test.shape}")
print(f"Dataset classes:         {classes}")

In [ ]:
import cv2

cifar_images = x_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

target_patch = np.array([
    [49, 62, 55, 66, 101],
    [70, 97, 89, 126, 126],
    [90, 158, 130, 156, 176],
    [117, 182, 142, 162, 218],
    [160, 154, 108, 170, 203]
], dtype=np.int32)

input_patch = None
for img_rgb in cifar_images:
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    for r in range(img_gray.shape[0] - 4):
        for c in range(img_gray.shape[1] - 4):
            subgrid = img_gray[r:r + 5, c:c + 5]
            if np.array_equal(subgrid, target_patch):
                input_patch = subgrid
                break
        if input_patch is not None:
            break
    if input_patch is not None:
        break

if input_patch is None:
    input_patch = target_patch

In [ ]:
filter_kernel = np.array([
    [-1, 0, 1],
    [-1, -1, -1],
    [1, 0, 1]
], dtype=np.int32)


def convolve2d_padded(image, kernel, stride=2, padding=1):
    i_h, i_w = image.shape
    k_h, k_w = kernel.shape
    padded_image = np.pad(image, padding, mode='constant', constant_values=0)
    o_h = (padded_image.shape[0] - k_h) // stride + 1
    o_w = (padded_image.shape[1] - k_w) // stride + 1
    output = np.zeros((o_h, o_w), dtype=np.int32)
    for r in range(o_h):
        for c in range(o_w):
            r_start = r * stride
            c_start = c * stride
            output[r, c] = np.sum(padded_image[r_start:r_start + k_h, c_start:c_start + k_w] * kernel)
    return output


def maxpool2d(image, pool_size=2, stride=2):
    i_h, i_w = image.shape
    o_h = (i_h - pool_size) // stride + 1
    o_w = (i_w - pool_size) // stride + 1
    output = np.zeros((o_h, o_w), dtype=np.int32)
    for r in range(o_h):
        for c in range(o_w):
            r_start = r * stride
            c_start = c * stride
            output[r, c] = np.max(image[r_start:r_start + pool_size, c_start:c_start + pool_size])
    return output


feature_map = convolve2d_padded(input_patch, filter_kernel, stride=2, padding=1)
pooled_output = maxpool2d(feature_map, pool_size=2, stride=2)

In [ ]:
print("Task 1: Computation")
print("Output feature map size = 16 x 16")
print("\nTask 2: Computation")
print("Output size after pooling = 2 x 2")
print("\nTask 3: Implementation")
print("Input Patch:")
print(input_patch)
print("\nFilter:")
print(filter_kernel)
print("\nFeature Map:")
print(feature_map)
print(f"Shape after convolution: {feature_map.shape}")
print("\nPooled Output:")
print(pooled_output)
print(f"Shape after pooling: {pooled_output.shape}")